In [ ]:
import nibabel as nib
import numpy as np
import os
from PIL import Image

def convert_patient_to_png_slices(patient_folder_path, output_folder, modality='t2'):
    """
    Extract slices at 25%, 38%, 50%, 62%, 75% along the z-axis from the T2 and segmentation
    NIfTI files, z-score normalise T2 on brain voxels, and save all 6 slices as PNGs.

    Args:
        patient_folder_path: Path to the patient folder
        output_folder: Root output directory; a subfolder per patient is created here
        modality: MRI modality to process. Default is 't2'
    """
    try:
        patient_id = os.path.basename(patient_folder_path)
        input_nii_path = os.path.join(patient_folder_path, f"{patient_id}_{modality}.nii")
        input_seg_path = os.path.join(patient_folder_path, f"{patient_id}_seg.nii")

        image = nib.load(input_nii_path).get_fdata()
        seg   = nib.load(input_seg_path).get_fdata()

        # Z-score normalise T2 on brain voxels only
        mask_brain = image > 0
        if mask_brain.any():
            mean, std = image[mask_brain].mean(), image[mask_brain].std()
            normalised = np.zeros_like(image, dtype=np.float32)
            normalised[mask_brain] = (image[mask_brain] - mean) / (std + 1e-8)
        else:
            normalised = image.copy()

        output_sub = os.path.join(output_folder, patient_id)
        os.makedirs(output_sub, exist_ok=True)

        depth = image.shape[2]
        quartiles = {
            'q25': int(depth * 0.25),
            'q38': int(depth * 0.38),
            'q50': int(depth * 0.50),
            'q62': int(depth * 0.62),
            'q75': int(depth * 0.75),
        }

        for qname, z in quartiles.items():
            # T2 slice: min-max scale to [0, 255] uint8 for PNG storage
            t2_slice = normalised[:, :, z].T  # transpose matches display convention
            lo, hi = t2_slice.min(), t2_slice.max()
            t2_uint8 = ((t2_slice - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
            Image.fromarray(t2_uint8).save(
                os.path.join(output_sub, f"{patient_id}_t2_{qname}.png"))

            # Seg slice: class labels (0/1/2/4) stored directly as uint8
            seg_slice = seg[:, :, z].T.astype(np.uint8)
            Image.fromarray(seg_slice).save(
                os.path.join(output_sub, f"{patient_id}_seg_{qname}.png"))

        return True

    except Exception as e:
        print(f"Failed to convert: {patient_folder_path}. Got error: {e}")
        return False


# Example usage:
# convert_patient_to_png_slices('/path/to/BraTS20_Training_001', '/path/to/output_folder')

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import os

def display_middle_slice_nii(patient_path, modality='t2'):
    """
    Opens NIfTI files and displays the middle slice of the scan and the mask.
    
    Args:
        patient_path: Path to the patient folder containing NIfTI files
        modality: Which modality to display ('flair', 't1', 't1ce', 't2'). Default is 't2'.
    """
    patient_id = os.path.basename(patient_path)
    
    # Load the specific modality
    modality_file = os.path.join(patient_path, f"{patient_id}_{modality}.nii")
    img = nib.load(modality_file).get_fdata()
    print(f"Image shape: {img.shape}")  # Print shape to verify dimensions
    
    # Load the segmentation mask
    seg_file = os.path.join(patient_path, f"{patient_id}_seg.nii")
    mask = nib.load(seg_file).get_fdata()
    
    # Determine the middle slice index along the depth axis
    # BraTS is typically 155 slices deep (axis 2)
    mid_idx = img.shape[2] // 2
    
    # Extract the middle slice
    slice_img = img[:, :, mid_idx]
    slice_mask = mask[:, :, mid_idx]
    
    # Plotting
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.title(f"Middle Slice ({mid_idx}) - {modality.upper()}")
    im = plt.imshow(slice_img.T, cmap='gray', origin='lower')
    plt.colorbar(im, label='Magnitudes')
    plt.axis('on')

    plt.subplot(1, 2, 2)
    plt.title(f"Middle Slice ({mid_idx}) - Mask")
    im = plt.imshow(slice_mask.T, cmap='jet', origin='lower') 
    plt.colorbar(im, label='Mask Classes')
    plt.axis('on')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Displayed middle slice {mid_idx} from {patient_id} - {modality.upper()}")

In [ ]:
import glob
import os
from pathlib import Path

cwd = Path.cwd()
print("Current working directory:", cwd) # /path/to/home
root_path = Path("/path/to/BrainWear_Kareem")

# Set paths
brats_root = root_path/"BraTS2020_TrainingData"/"MICCAI_BraTS2020_TrainingData"
search_pattern = str(brats_root/"BraTS20_Training_*")

patient_folders = glob.glob(search_pattern)

# Check all patient folders are present
assert len(patient_folders) == 369, f"Expected 369 patient folders in the BraTS20_TrainingData directory, found {len(patient_folders)}."
print(f"Found {len(patient_folders)} patient folders. All present")

# Check patient folder contains expected files
expected_files = ['flair', 't1', 't1ce', 't2', 'seg']
sample_patient = brats_root/"BraTS20_Training_004" # Random patient
print(f"Sample patient: {sample_patient}")
for mod in expected_files:
    file_path = os.path.join(sample_patient, f"{os.path.basename(sample_patient)}_{mod}.nii")
    assert os.path.exists(file_path), f"Expected file {file_path} not found in {sample_patient}."
print(f"Sample patient folder {os.path.basename(sample_patient)} contains all expected files.")

# Check output path exists
output_path = root_path/'Processed_BraTS2020_TrainingData_PNG_new_norm'
os.makedirs(output_path, exist_ok=True)
assert os.path.exists(output_path), f"Output path {output_path} does not exist."
print(f"Output path {output_path} is valid.")

In [ ]:
# Display middle slice of sample patient to verify NIfTI files are readable
display_middle_slice_nii(sample_patient, modality='t2')

In [ ]:
# Convert all patient folders — extract quartile PNG slices for T2 and segmentation
failures = []
count = None       # Number of patient folders to convert for testing (set to None for all)
if count is not None:
    print(f"Converting the first {count} patient folders for testing...")
    for patient_folder in patient_folders[:count]:
        patient_id = os.path.basename(patient_folder)
        if not convert_patient_to_png_slices(patient_folder, output_path, modality='t2'):
            failures.append(patient_id)
else:
    count = len(patient_folders)
    print("Converting all patient folders...")
    for i, patient_folder in enumerate(patient_folders):
        patient_id = os.path.basename(patient_folder)
        if not convert_patient_to_png_slices(patient_folder, output_path, modality='t2'):
            failures.append(patient_id)
        if i % 10 == 0:
            print(f"{i}/{count} patients processed")

if failures:
    raise RuntimeError(f"Conversions failed for patient ids: {', '.join(failures)}")

# Verify each patient folder contains exactly 10 PNGs (5 T2 + 5 seg)
processed_folders = glob.glob(os.path.join(output_path, 'BraTS20_Training_*'))
assert len(processed_folders) == count, f"Expected {count} patient folders, but found {len(processed_folders)}."
for folder in processed_folders:
    pngs = glob.glob(os.path.join(folder, '*.png'))
    assert len(pngs) == 10, f"Expected 10 PNGs in {folder}, found {len(pngs)}."
print("All patient folders were successfully processed (10 PNGs each).")

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def inspect_patient_pngs(patient_output_path):
    """
    Load and display the 10 quartile PNGs (T2 + seg) for a processed patient.
    """
    import glob, os
    pngs = sorted(glob.glob(os.path.join(patient_output_path, '*.png')))
    print(f"Found {len(pngs)} PNGs in {patient_output_path}")
    fig, axes = plt.subplots(2, 5, figsize=(15, 8))
    for ax, png_path in zip(axes.flat, pngs):
        arr = np.array(Image.open(png_path))
        label = os.path.basename(png_path)
        cmap = 'jet' if '_seg_' in label else 'gray'
        ax.imshow(arr, cmap=cmap, origin='lower')
        ax.set_title(label, fontsize=8)
        ax.axis('off')
        print(f"  {label}: shape={arr.shape}, min={arr.min()}, max={arr.max()}, dtype={arr.dtype}")
    plt.tight_layout()
    plt.show()

In [ ]:
sample_patient_output = output_path/"BraTS20_Training_004"
inspect_patient_pngs(sample_patient_output)

In [ ]:
import numpy as np
from PIL import Image

# Quick sanity check: print pixel stats for all 10 PNGs of a sample patient
import glob, os
sample_pngs = sorted(glob.glob(os.path.join(sample_patient_output, '*.png')))
for png_path in sample_pngs:
    arr = np.array(Image.open(png_path))
    label = os.path.basename(png_path)
    if '_seg_' in label:
        print(f"{label}: shape={arr.shape}, unique labels={np.unique(arr).tolist()}")
    else:
        print(f"{label}: shape={arr.shape}, min={arr.min()}, max={arr.max()}")